In [4]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langgraph.graph import StateGraph, START, END
from huggingface_hub import login
from langchain_core.messages import BaseMessage,HumanMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    max_new_tokens=512,
    temperature=0.7
)

model = ChatHuggingFace(llm=llm)

In [3]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [ ]:
def generate_joke(state:JokeState):
    prompt=f"Generate a hilarious joke on the topic {state["topic"]}"
    response=model.invoke(prompt).content
    return {'joke':response}
    

In [6]:
def explain(state:JokeState):
    prompt=f"give an explaination for the joke {state["joke"]}"
    response=model.invoke(prompt).content
    return {'explaination':response}

In [14]:
graph=StateGraph(JokeState)
graph.add_node("generate_joke",generate_joke)
graph.add_node("explain",explain)

graph.add_edge(START,"generate_joke")
graph.add_edge("generate_joke","explain")
graph.add_edge("explain",END)

checkpointer=InMemorySaver()
workflow=graph.compile(checkpointer=checkpointer)


In [16]:
config1={"configurable":{"thread_id":"1"}}
topic="pizza"
initial_state={'topic':topic}
workflow.invoke(initial_state,config=config1)


{'topic': 'pizza',
 'joke': "Here's one:\n\nWhy was the pizza in therapy?\n\nBecause it was feeling crusty and had a lot of dough to work through!\n\n(get it?)"}

In [19]:
workflow.get_state(config1)
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in therapy?\n\nBecause it was feeling crusty and had a lot of dough to work through!\n\n(get it?)"}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a5205-8829-67d0-8002-9cbe214bc5a6'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-31T09:42:56.768591+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a5205-3d92-6e3d-8001-37366de67f62'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in therapy?\n\nBecause it was feeling crusty and had a lot of dough to work through!\n\n(get it?)"}, next=('explain',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1a5205-3d92-6e3d-8001-37366de67f62'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-31T09:42:48.947442+00:00'